# NB10 - Neural Collapse
## Background
Neural collapse (Papyan, Han & Donoho, 2020) is a geometric phenomenon that occurs in the terminal phase of training (TPT) — the period after training loss has reached near zero. Four properties emerge simultaneously: (NC1) within-class variability vanishes; (NC2) class means form an Equiangular Tight Frame (ETF) — maximally separated in all directions; (NC3) the linear classifier weights align with the class means; (NC4) the classifier becomes the nearest-class-mean classifier.

This ETF geometry is remarkable: it is provably optimal for linear separability and has maximum pairwise angles between classes. Neural networks converge to this structure regardless of the training data distribution, as if they discover an optimal geometric arrangement.

## What You'll Learn
- Simulate and visualize neural collapse in a 3-class problem
- ETF geometry: equiangular, tight frame, maximum separation
- NC1-NC4 metrics: within-class variability, cosine similarity alignment
- How neural collapse relates to generalization and few-shot learning

## Why This Matters
Neural collapse provides a theoretical explanation for why deep networks generalize even in the overparameterized regime. It also motivates neural collapse regularizers that force ETF structure earlier in training, and explains the success of nearest-prototype classifiers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# ── ETF: Equiangular Tight Frame ──────────────────────────────────────────
def make_etf(n_classes: int, d: int) -> np.ndarray:
    """Construct an ETF in R^d for n_classes classes.
    ETF: mu_i^T mu_j = -1/(K-1) for i!=j, ||mu_i|| = 1.
    Uses a random rotation of a simplex ETF.
    """
    K = n_classes
    # Simplex ETF construction
    etf = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            if i == j:
                etf[i, j] = 1.0
            else:
                etf[i, j] = -1.0 / (K - 1)
    # Embed in higher-dimensional space via random rotation
    if d > K:
        Q, _ = np.linalg.qr(np.random.randn(d, K))
        return (Q @ etf.T).T  # (K, d)
    return etf[:, :d]

# ── Simulate feature collapse toward ETF ──────────────────────────────────
def simulate_neural_collapse(n_classes: int = 5, d: int = 3,
                              n_per_class: int = 50,
                              collapse_level: float = 0.0,
                              seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """Simulate features at collapse_level 0.0=random, 1.0=full collapse."""
    np.random.seed(seed)
    etf_means = make_etf(n_classes, d)
    features, labels = [], []
    for c in range(n_classes):
        # Within-class variance decreases as collapse_level -> 1
        within_var = (1.0 - collapse_level) * 0.5
        # Interpolate between random center and ETF center
        random_center = np.random.randn(d) * 0.3
        center = collapse_level * etf_means[c] + (1-collapse_level) * random_center
        class_feats = center + np.random.randn(n_per_class, d) * within_var
        features.append(class_feats)
        labels.extend([c] * n_per_class)
    return np.vstack(features), np.array(labels)

# ── NC1: Within-class variability ──────────────────────────────────────────
def nc1_metric(features: np.ndarray, labels: np.ndarray) -> float:
    """NC1: Trace(Sigma_W) / Trace(Sigma_B) (lower = more collapsed)."""
    classes = np.unique(labels)
    global_mean = features.mean(axis=0)
    Sigma_W = np.zeros((features.shape[1], features.shape[1]))
    Sigma_B = np.zeros_like(Sigma_W)
    for c in classes:
        mask = labels == c
        mu_c = features[mask].mean(axis=0)
        X_c  = features[mask] - mu_c
        Sigma_W += X_c.T @ X_c
        diff = (mu_c - global_mean).reshape(-1, 1)
        Sigma_B += mask.sum() * (diff @ diff.T)
    tr_W = np.trace(Sigma_W)
    tr_B = np.trace(Sigma_B)
    return tr_W / (tr_B + 1e-10)

# ── NC2: ETF alignment ────────────────────────────────────────────────────
def nc2_metric(features: np.ndarray, labels: np.ndarray) -> float:
    """NC2: How close are pairwise cosine similarities to -1/(K-1)?"""
    classes = np.unique(labels)
    K = len(classes)
    means = np.array([features[labels==c].mean(0) for c in classes])
    # Normalize
    means = means / (np.linalg.norm(means, axis=1, keepdims=True) + 1e-10)
    target = -1.0 / (K - 1)
    deviations = []
    for i in range(K):
        for j in range(i+1, K):
            cos_sim = float(means[i] @ means[j])
            deviations.append(abs(cos_sim - target))
    return np.mean(deviations)

# ── Show collapse progression ──────────────────────────────────────────────
levels = [0.0, 0.3, 0.6, 0.9, 1.0]
nc1_scores, nc2_scores = [], []

for level in levels:
    feats, labels = simulate_neural_collapse(n_classes=5, d=3,
                                              collapse_level=level)
    nc1_scores.append(nc1_metric(feats, labels))
    nc2_scores.append(nc2_metric(feats, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(levels, nc1_scores, 'b-o', linewidth=2)
axes[0].set_title('NC1: Within-class Variability Ratio')
axes[0].set_xlabel('Collapse level'); axes[0].set_ylabel('Tr(Sigma_W)/Tr(Sigma_B)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(levels, nc2_scores, 'r-o', linewidth=2)
axes[1].set_title('NC2: ETF Alignment Deviation')
axes[1].set_xlabel('Collapse level'); axes[1].set_ylabel('Mean deviation from -1/(K-1)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_10_neural_collapse_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

print('Neural collapse progression:')
print(f'  Level 0.0: NC1={nc1_scores[0]:.3f}, NC2={nc2_scores[0]:.4f}')
print(f'  Level 1.0: NC1={nc1_scores[-1]:.3f}, NC2={nc2_scores[-1]:.4f}')
print()
print('At full collapse: NC1~0 (zero within-class variance), NC2~0 (perfect ETF)')

In [ ]:
# ── 3D visualization of ETF structure ─────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(16, 6))

for idx, (level, title) in enumerate([
    (0.0, 'Random (no collapse)'),
    (0.5, 'Partial collapse'),
    (1.0, 'Full collapse (ETF)'),
]):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    feats, labels = simulate_neural_collapse(5, 3, 30, collapse_level=level, seed=42)
    colors = plt.cm.tab10(np.linspace(0, 0.5, 5))
    for c in range(5):
        mask = labels == c
        ax.scatter(feats[mask, 0], feats[mask, 1], feats[mask, 2],
                   c=[colors[c]], s=20, alpha=0.7, label=f'C{c}')
        mu = feats[mask].mean(0)
        ax.scatter(*mu, c='black', s=80, marker='*', zorder=10)
    ax.set_title(f'{title}\nNC1={nc1_metric(feats,labels):.3f}')
    ax.set_xlabel('d1'); ax.set_ylabel('d2'); ax.set_zlabel('d3')

plt.tight_layout()
plt.savefig(f'{BASE}/s30_10_neural_collapse_3d.png', dpi=100, bbox_inches='tight')
plt.show()
print('ETF geometry: all class means pairwise equidistant (angle = arccos(-1/(K-1)))')
K = 5
import math
target_angle = math.degrees(math.acos(-1/(K-1)))
print(f'For K={K} classes: target pairwise angle = {target_angle:.1f} degrees')